In [0]:
from pyspark.sql import functions as f,Window
import re
from datetime import datetime
from pyspark.sql.types import *

c_schema = StructType([
    StructField('customer_id',IntegerType(),False),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType()),
    StructField('load_date',StringType()),
])

a_schema = StructType([
    StructField('account_id',IntegerType()),
    StructField('customer_id',IntegerType(),False),
    StructField('account_type',StringType()),
    StructField('branch_city',StringType()),
    StructField('status',StringType()),
    StructField('start_date',StringType()),
    StructField('end_date',StringType()),
    StructField('is_current',StringType())
])

t_schema = StructType(
    [
        StructField('transaction_id',IntegerType()),
        StructField('account_id',IntegerType()),
        StructField('transaction_type',StringType()),
        StructField('amount',IntegerType()),
        StructField('transaction_date',StringType())
    ]
)

source_path = '/Volumes/workspace/madstage/stage/stageInc/'
files = dbutils.fs.ls(source_path)

customer_files = []
account_files = []
transcation_files = []

for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.path)
    if file.name.startswith('customer'):
        if match:
            file_date = datetime.strptime(match.group(1),"%Y-%m-%d")
            customer_files.append((file_date,file.path))
    elif file.name.startswith('account'):
        if match:
            file_date = datetime.strptime(match.group(1),"%Y-%m-%d")
            account_files.append((file_date,file.path))
    elif file.name.startswith('transaction'):
        if match:
            file_date = datetime.strptime(match.group(1),"%Y-%m-%d")
            transcation_files.append((file_date,file.path))

if customer_files:
    latest_file = max(customer_files,key=lambda x:x[0])
    print(latest_file)
    b_customers =(spark.read.format('csv').schema(c_schema).option('header','true').load(latest_file[1]))
if account_files:
    latest_file = max(account_files,key=lambda x:x[0])
    print(latest_file)
    b_accounts = (spark.read.format('csv').schema(a_schema).option('header','true').load(latest_file[1]))
if transcation_files:
    latest_file = max(transcation_files,key=lambda x:x[0])
    print(latest_file)
    b_transactions = (spark.read.format('csv').schema(t_schema).option('header','true').load(latest_file[1]))

b_customers = b_customers.select('customer_id','customer_name','city','customer_segment','email','phone','is_current')
b_accounts = b_accounts.select('account_id','customer_id','account_type','branch_city','status','is_current')
b_transactions = b_transactions.select('transaction_id','account_id','transaction_type','amount','transaction_date')

# drop null
b_customers = b_customers.dropna(subset='customer_id')
b_accounts = b_accounts.dropna(subset='customer_id')
b_transactions = b_transactions.dropna(subset='account_id')

# drop duplicates
b_customers = b_customers.dropDuplicates()
b_accounts = b_accounts.dropDuplicates()
b_transactions = b_transactions.dropDuplicates()

# null handling
b_customers = b_customers.fillna(
    {
        'customer_name':'Unknown',
        'city':'Unknown',
        'customer_segment':'Unknown',
        'email':'Unknown',
        'phone':0,
        'is_current':'Y'
    }
)

b_accounts = b_accounts.fillna({
    'customer_id':0,
    'account_type':'Unknown',
    'branch_city':'Unknown',
    'status':'Unknown',
    'is_current':'Y'
})

b_transactions = b_transactions.fillna({
    'transaction_type':'Unknown',
    'amount':0
})

# standardization
phone_regex = r'[^0-9]'
b_customers = b_customers.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.lower(f.trim(f.col('email')))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,''))

b_accounts = b_accounts.withColumn('account_type',f.initcap(f.trim(f.col('account_type')))).withColumn('branch_city',f.initcap(f.trim(f.col('branch_city')))).withColumn('status',f.initcap(f.trim('status')))

b_transactions = b_transactions.withColumn('transaction_type',f.initcap(f.trim(f.col('transaction_type'))))

# data validation
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation =(f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)
amount_validation = (f.col('amount') >= 0)
status = (f.col('status')=='Active')

b_customers = b_customers.withColumn('dataValidation',f.when((email_validation)&(phone_validation),'Pass').otherwise('Fail'))
b_accounts = b_accounts.withColumn('dataValidation',f.when(status,'Pass').otherwise('Fail'))
b_transactions = b_transactions.withColumn('dataValidation',f.when(amount_validation,'Pass').otherwise('Fail'))

# audit column
b_customers = b_customers.withColumn('updated_customer_dt',f.lit('9999-12-31').cast('string')).withColumn('created_customer_dt',f.current_date().cast('string'))
b_accounts = b_accounts.withColumn('updated_account_dt',f.lit('9999-12-31').cast('string')).withColumn('created_account_dt',f.current_date().cast('string'))
b_transactions = b_transactions.drop('data_validation').withColumn('is_current',f.lit('Y'))

# filter valid records
b_customers = b_customers.filter(f.col('dataValidation')=='Pass').drop('dataValidation')
b_accounts = b_accounts.filter(f.col('dataValidation')=='Pass').drop('dataValidation')
b_transactions = b_transactions.filter(f.col('dataValidation')=='Pass').drop('dataValidation')

b_customers.createOrReplaceTempView('b_customers')
b_accounts.createOrReplaceTempView('b_accounts')
b_transactions.createOrReplaceTempView('b_transactions')

customers = spark.read.table('workspace.madstage.customers')
accounts = spark.read.table('workspace.madstage.accounts')
transactions = spark.read.table('workspace.madstage.transactions')

customers.printSchema()
# accounts.printSchema()
# transactions.printSchema()

b_customers.printSchema()
# b_accounts.printSchema()
# b_transactions.printSchema()

# scd1 for customers
spark.sql("""
          merge into workspace.madstage.customers as target
          using b_customers as source
          on target.customer_id = source.customer_id
          when matched then
          update set
          target.customer_name = source.customer_name,
          target.city = source.city,
          target.customer_segment = source.customer_segment,
          target.email = source.email,
          target.phone = source.phone,
          target.updated_customer_dt = source.updated_customer_dt,
          target.created_customer_dt = source.created_customer_dt
          when not matched then
          insert *
          """)

# scd1 for accounts
spark.sql("""
          merge into workspace.madstage.accounts as target
          using b_accounts as source
          on target.account_id = source.account_id
          when matched then
          update set
          target.account_type = source.account_type,
          target.branch_city = source.branch_city,
          target.status = source.status,
          target.updated_account_dt = source.updated_account_dt,
          target.created_account_dt = source.created_account_dt
          when not matched then
          insert *
          """)

# scd1 for transactions
spark.sql("""
          MERGE INTO workspace.madstage.transactions AS target
          USING b_transactions AS source
          ON target.transaction_id = source.transaction_id
          WHEN MATCHED THEN
          UPDATE SET
          target.account_id = source.account_id,
          target.transaction_type = source.transaction_type,
          target.amount = source.amount,
          target.transaction_date = source.transaction_date,
          target.is_current = source.is_current
          WHEN NOT MATCHED THEN
          INSERT *
""")

print('Incremental ETL completed')
            